In [7]:
# Importanção de bibliotecas
import ee
import pandas as pd

In [ ]:
# 1. Autenticação e Iniciaçialização - GEE
ee.Authenticate()
ee.Initialize(project = '') # adicione seu projeto


Successfully saved authorization token.


In [36]:
# 2. Definição do local a partir de coordenadas ou shapes
poligonos = ee.FeatureCollection([
    # Fazenda / Talhão em Sorriso - MT (Eixo da BR-163 - Capital da Soja)
    ee.Feature(
        ee.Geometry.Polygon([[
            [-55.7520, -12.4850],
            [-55.7050, -12.4850],
            [-55.7050, -12.5350],
            [-55.7520, -12.5350],
            [-55.7520, -12.4850]
        ]]), 
        {
            'farm_id': 'Fazenda_Sorriso_MT',
            'municipio': 'Sorriso',
            'uf': 'MT',
            'crop': 'Soja / Milho Safrinha'
        }
    ),
    # Fazenda / Talhão em Luís Eduardo Magalhães - BA (Oeste Baiano / MATOPIBA)
    ee.Feature(
        ee.Geometry.Polygon([[
            [-45.9400, -12.1800],
            [-45.8900, -12.1800],
            [-45.8900, -12.2300],
            [-45.9400, -12.2300],
            [-45.9400, -12.1800]
        ]]), 
        {
            'farm_id': 'Fazenda_LEM_BA',
            'municipio': 'Luis Eduardo Magalhaes',
            'uf': 'BA',
            'crop': 'Algodao / Soja'
        }
    ),
    # Fazenda / Talhão em Rio Verde - GO (Sudoeste Goiano)
    ee.Feature(
        ee.Geometry.Polygon([[
            [-50.9600, -17.7500],
            [-50.9000, -17.7500],
            [-50.9000, -17.8100],
            [-50.9600, -17.8100],
            [-50.9600, -17.7500]
        ]]), 
        {
            'farm_id': 'Fazenda_RioVerde_GO',
            'municipio': 'Rio Verde',
            'uf': 'GO',
            'crop': 'Soja'
        }
    )
])

In [40]:
# 3. Filtragem da coleção (CHIRPS Daily: 1981 - Presente)
start_date = '2026-01-01'
end_date = '2026-01-31'

chirps = (
    ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
    .filterDate(start_date, end_date)
    .filterBounds(poligonos)
    .select('precipitation')
)

In [41]:
# 4. Função de extração zonal diária
def process_daily(image):
    date_str = image.date().format('YYYY-MM-dd')

    daily_reduction = image.reduceRegions(
        collection = poligonos,
        reducer = ee.Reducer.mean().setOutputs(['precipitation_mm']),
        scale = 5666 # Resolução espacial nativa do sensor CHIRPS (~0.05°)
    )

    return daily_reduction.map(lambda feat: feat.set('date', date_str))

daily_chuva = chirps.map(process_daily).flatten()

In [42]:
# 5. Extração e montagem do dataframe
data = daily_chuva.getInfo()

records = [
    {
        'farm_id': f['properties'].get('farm_id'),
        'municipio': f['properties'].get('municipio'),
        'uf': f['properties'].get('uf'),
        'crop': f['properties'].get('crop'),
        'date': f['properties'].get('date'),
        'precipitation_mm': round(f['properties'].get('precipitation_mm', 0.0), 2)
    }
    for f in data['features']
]

df_chuva = pd.DataFrame(records)
df_chuva['date'] = pd.to_datetime(df_chuva['date'])
df_chuva = df_chuva.sort_values(by=['farm_id', 'date']).reset_index(drop=True)

In [43]:
df_chuva.head(10)

,farm_id,municipio,uf,crop,date,precipitation_mm
0,Fazenda_LEM_BA,Luis Eduardo Magalhaes,BA,Algodao / Soja,2026-01-01,4.71
1,Fazenda_LEM_BA,Luis Eduardo Magalhaes,BA,Algodao / Soja,2026-01-02,9.43
2,Fazenda_LEM_BA,Luis Eduardo Magalhaes,BA,Algodao / Soja,2026-01-03,2.62
3,Fazenda_LEM_BA,Luis Eduardo Magalhaes,BA,Algodao / Soja,2026-01-04,0.00
4,Fazenda_LEM_BA,Luis Eduardo Magalhaes,BA,Algodao / Soja,2026-01-05,0.00
5,Fazenda_LEM_BA,Luis Eduardo Magalhaes,BA,Algodao / Soja,2026-01-06,0.17
6,Fazenda_LEM_BA,Luis Eduardo Magalhaes,BA,Algodao / Soja,2026-01-07,0.00
7,Fazenda_LEM_BA,Luis Eduardo Magalhaes,BA,Algodao / Soja,2026-01-08,0.00
8,Fazenda_LEM_BA,Luis Eduardo Magalhaes,BA,Algodao / Soja,2026-01-09,0.00
9,Fazenda_LEM_BA,Luis Eduardo Magalhaes,BA,Algodao / Soja,2026-01-10,0.00
